# ToMnet Figure 5 Visualization

This notebook reproduces the visualizations from Figure 5 of the "Machine Theory of Mind" paper.

Figure 5 shows:
- (a) Prediction accuracy vs number of past observations for goal-directed agents
- (b) 2D character embeddings colored by reward preferences
- (c) Policy predictions showing goal-directed movement patterns
- (d) Inference of cost-reward balance from single trajectories

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch
from collections import defaultdict
import pandas as pd
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

# Set style
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Object colors for visualization
OBJECT_COLORS = ["red", "blue", "green", "yellow"]
ACTION_ARROWS = {0: "↑", 1: "↓", 2: "←", 3: "→", 4: "•"}  # up, down, left, right, stay

## Load Evaluation Results

In [ ]:
# Load evaluation results
with open("result/evaluation_results_figure5.pkl", "rb") as f:
    results = pickle.load(f)

print("Available keys:", list(results.keys()))

# Load original dataset for additional analysis
with open("data/figure5_data.pkl", "rb") as f:
    dataset = pickle.load(f)

print(f"Dataset contains {len(dataset['data'])} samples")
print(f"Number of agents: {dataset['meta']['n_agents']}")

## Figure 5a: Prediction Accuracy vs Number of Past Observations

In [ ]:
    plt.tight_layout()
    plt.savefig("result/figure5a_accuracy_vs_n_past.png", dpi=300, bbox_inches="tight")
    plt.show()

## Figure 5b: Character Embeddings Colored by Reward Preferences

In [ ]:
    plt.tight_layout()
    plt.savefig("result/figure5b_character_embeddings.png", dpi=300, bbox_inches="tight")
    plt.show()

## Figure 5c: Policy Predictions and Goal Inference

In [ ]:
    plt.tight_layout()
    plt.savefig("result/figure5c_policy_predictions.png", dpi=300, bbox_inches="tight")
    plt.show()

## Figure 5d: Cost-Reward Tradeoff Inference

In [ ]:
    plt.tight_layout()
    plt.savefig("result/figure5d_cost_reward_analysis.png", dpi=300, bbox_inches="tight")
    plt.show()

## Comprehensive Figure 5 Reproduction

In [ ]:
    plt.tight_layout()
    plt.savefig("result/figure5_reproduction.png", dpi=300, bbox_inches="tight")
    plt.show()

## Summary Statistics and Analysis

In [ ]:
def print_figure5_summary(results, dataset):
    """Print comprehensive summary statistics"""
    print("=== FIGURE 5 SUMMARY STATISTICS ===")
    print()

    # Basic performance metrics
    print("Performance Metrics:")
    print("-" * 40)
    print(f"Overall action accuracy: {results['action_accuracy']:.3f}")

    accuracy_by_n_past = results["accuracy_by_n_past"]
    if accuracy_by_n_past:
        min_n_past = min(accuracy_by_n_past.keys())
        max_n_past = max(accuracy_by_n_past.keys())
        min_acc = accuracy_by_n_past[min_n_past]
        max_acc = accuracy_by_n_past[max_n_past]
        improvement = max_acc - min_acc

        print(f"Accuracy with {min_n_past} past episodes: {min_acc:.3f}")
        print(f"Accuracy with {max_n_past} past episodes: {max_acc:.3f}")
        print(f"Improvement from observation: {improvement:.3f}")
        print(f"Relative improvement: {improvement/min_acc*100:.1f}%")

    print()

    # Agent diversity analysis
    print("Agent Diversity:")
    print("-" * 40)

    agent_rewards = results["agent_rewards"]
    n_unique_agents = len(agent_rewards)
    print(f"Number of unique agents: {n_unique_agents}")

    # Analyze reward preferences
    preferred_objects = []
    reward_entropies = []

    for agent_id, rewards in agent_rewards.items():
        preferred_obj = np.argmax(rewards)
        preferred_objects.append(preferred_obj)

        # Compute entropy
        entropy = -np.sum(rewards * np.log(rewards + 1e-10))
        reward_entropies.append(entropy)

    # Object preference distribution
    from collections import Counter

    pref_counts = Counter(preferred_objects)
    print("Object preference distribution:")
    for obj_id in range(4):
        count = pref_counts.get(obj_id, 0)
        percentage = count / len(preferred_objects) * 100
        print(f"  Object {obj_id+1}: {count} agents ({percentage:.1f}%)")

    print(f"Mean reward entropy: {np.mean(reward_entropies):.3f}")
    print(f"Std reward entropy: {np.std(reward_entropies):.3f}")

    print()

    # Cost analysis
    print("Cost Analysis:")
    print("-" * 40)

    agent_costs = {}
    for sample in dataset["data"]:
        agent_id = sample["agent_id"]
        if agent_id not in agent_costs:
            agent_costs[agent_id] = sample["movement_cost"]

    costs = list(agent_costs.values())
    low_cost_agents = sum(1 for c in costs if c <= 0.1)
    high_cost_agents = sum(1 for c in costs if c >= 0.4)

    print(
        f"Low cost agents (≤0.1): {low_cost_agents} ({low_cost_agents/len(costs)*100:.1f}%)"
    )
    print(
        f"High cost agents (≥0.4): {high_cost_agents} ({high_cost_agents/len(costs)*100:.1f}%)"
    )
    print(f"Mean movement cost: {np.mean(costs):.3f}")
    print(f"Cost range: {np.min(costs):.3f} - {np.max(costs):.3f}")

    print()

    # Dataset statistics
    print("Dataset Statistics:")
    print("-" * 40)
    print(f"Total samples: {len(dataset['data'])}")
    print(f"Number of agents: {dataset['meta']['n_agents']}")
    print(f"Episodes per agent: {dataset['meta']['n_episodes_per_agent']}")
    print(f"High cost agent ratio: {dataset['meta']['high_cost_ratio']}")

    # Analyze n_past distribution
    n_past_values = [sample["n_past"] for sample in dataset["data"]]
    print(f"N_past range: {np.min(n_past_values)} - {np.max(n_past_values)}")
    print(f"Mean N_past: {np.mean(n_past_values):.1f}")


print_figure5_summary(results, dataset)

## Export Results

In [ ]:
# Export processed results for further analysis
import json

export_data = {
    "performance": {
        "overall_accuracy": results["action_accuracy"],
        "accuracy_by_n_past": results["accuracy_by_n_past"],
    },
    "embeddings": {
        "character_embeddings": results["character_embeddings"].tolist(),
        "agent_ids": results["agent_ids"].tolist(),
    },
    "agent_properties": {
        "rewards": {
            str(k): v.tolist() if hasattr(v, "tolist") else v
            for k, v in results["agent_rewards"].items()
        }
    },
    "dataset_info": dataset["meta"],
}

with open("result/figure5_processed_results.json", "w") as f:
    json.dump(export_data, f, indent=2)

print("Results exported to result/figure5_processed_results.json")
print("All visualization figures saved as PNG files in result/ directory")
print("\nNotebook execution completed!")